In [ ]:
from pyspark.sql.functions import col, avg, count, when, round as spark_round

# 1. Read from the Silver table
silver_df = spark.read.table("orders_weather_silver")

# 2. Business Question 1: Delivery impact by weather (Rain vs. No Rain)
weather_impact_df = silver_df.withColumn(
    "is_raining",
    when(col("precipitation_mm") > 0, "Yes").otherwise("No")
).groupBy("is_raining").agg(
    count("order_id").alias("total_orders"),
    spark_round(avg("delivery_duration_minutes"), 2).alias("avg_delivery_time_minutes")
)

# 3. Business Question 2: Restaurant Performance
restaurant_performance_df = silver_df.groupBy("restaurant_id").agg(
    count("order_id").alias("total_orders"),
    spark_round(avg("delivery_duration_minutes"), 2).alias("avg_delivery_time_minutes"),
    spark_round(avg("order_value_eur"), 2).alias("avg_order_value")
)

# 4. Save as Gold Delta Tables
weather_impact_df.write.format("delta").mode("overwrite").saveAsTable("gold_weather_impact")
restaurant_performance_df.write.format("delta").mode("overwrite").saveAsTable("gold_restaurant_performance")

print("Gold Layer Created! Weather Impact on Delivery:")
display(weather_impact_df)

print("Restaurant Performance Metrics:")
display(restaurant_performance_df)